In [1]:
import logging
from dataclasses import dataclass
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from rank_bm25 import BM25Okapi
import config

logger = logging.getLogger(__name__)

In [3]:
@dataclass
class RetrievedChunk:
    chunk_id: str
    domain: str
    section_id: str
    section_title: str
    parent_section: str
    text: str
    page: int | None
    score: float

In [10]:
def _to_chunk(payload: dict, score: float) -> RetrievedChunk:
    return RetrievedChunk(
        chunk_id=payload["chunk_id"],
        domain=payload["domain"],
        section_id=payload["section_id"],
        section_title=payload["section_title"],
        parent_section=payload.get("parent_section"),
        text=payload["text"],
        page=payload.get("page"),
        score=score,
    )

In [ ]:
_embed_model = None
_qdrant_client = None
_bm25_index = None
_bm25_chunks = None  

def get_embed_model() -> SentenceTransformer:
    global _embed_model
    if _embed_model is None:
        logger.info(f"Loading embedding model: {config.EMBED_MODEL}")
        _embed_model = SentenceTransformer(config.EMBED_MODEL)
    return _embed_model


def get_qdrant_client() -> QdrantClient:
    global _qdrant_client
    if _qdrant_client is None:
        _qdrant_client = QdrantClient(
            url=config.QDRANT_URL,
            api_key=config.QDRANT_API_KEY,
        )
    return _qdrant_client


def get_bm25_index(domain: str | None = None) -> tuple[BM25Okapi, list[dict]]:
    """
    Build BM25 index from Qdrant payloads on first call, cache thereafter.
    Optionally filter by domain.
    """
    global _bm25_index, _bm25_chunks

    if _bm25_index is None:
        logger.info("Building BM25 index from Qdrant...")
        client = get_qdrant_client()

        all_chunks = []
        offset = None

        while True:
            results, offset = client.scroll(
                collection_name=config.COLLECTION_NAME,
                limit=100,
                offset=offset,
                with_payload=True,
                with_vectors=False,
            )
            all_chunks.extend([r.payload for r in results])
            if offset is None:
                break

        _bm25_chunks = all_chunks
        corpus = [chunk["text"].lower().split() for chunk in all_chunks]
        _bm25_index = BM25Okapi(corpus)
        logger.info(f"BM25 index built over {len(all_chunks)} chunks")

    # Filter if domain specified
    if domain:
        filtered = [
            (i, c) for i, c in enumerate(_bm25_chunks)
            if c["domain"] == domain
        ]
        indices, chunks = zip(*filtered) if filtered else ([], [])
        corpus = [c["text"].lower().split() for c in chunks]
        return BM25Okapi(corpus), list(chunks), list(indices)

    return _bm25_index, _bm25_chunks, list(range(len(_bm25_chunks)))

In [9]:
bm25_index = get_bm25_index(domain="noise")

### Dense Search

In [ ]:
def dense_search(
    query: str,
    top_k: int = config.TOP_K_DENSE,
    domain: str | None = None,
) -> list[RetrievedChunk]:
    model = get_embed_model()
    client = get_qdrant_client()

    prefixed = f"Represent this sentence for searching relevant passages: {query}"
    vector = model.encode(prefixed, normalize_embeddings=True).tolist()

    query_filter = None
    if domain:
        query_filter = Filter(
            must=[FieldCondition(
                key="domain",
                match=MatchValue(value=domain)
            )]
        )

    results = client.query_points(
        collection_name=config.COLLECTION_NAME,
        query=vector,
        limit=top_k,
        query_filter=query_filter,
        with_payload=True,
    ).points

    return [_to_chunk(r.payload, r.score) for r in results]

In [25]:
dense_result = dense_search("can I rent my apartment for airbnb?", top_k=5)

In [26]:
dense_result

[RetrievedChunk(chunk_id='short_term_rental::547-2.13::0', domain='short_term_rental', section_id='547-2.13', section_title='Authority to suspend a licence or registration for immediate danger.', parent_section='ARTICLE 2', text='Short-term Rental Companies', page=1, score=0.59782666),
 RetrievedChunk(chunk_id='short_term_rental::547-3.7::0', domain='short_term_rental', section_id='547-3.7', section_title='Communications regarding short-term rental by-law.', parent_section='ARTICLE 3', text='Short-term Rental Operators', page=2, score=0.5934461),
 RetrievedChunk(chunk_id='short_term_rental::547-3.4::0', domain='short_term_rental', section_id='547-3.4', section_title='Creation and use of law enforcement accounts.', parent_section='ARTICLE 3', text="A. Every short-term rental company shall create operator and guest accounts on its platforms as requested by Municipal Licensing and Standards, to be used to investigate compliance with this chapter. 12 Editor's Note: This section of By-law 5

### Sparse Search

In [ ]:
def sparse_search(
    query: str,
    top_k: int = config.TOP_K_DENSE,
    domain: str | None = None,
) -> list[RetrievedChunk]:
    bm25, chunks, _ = get_bm25_index(domain)
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    # Get top_k indices by score
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    return [
        _to_chunk(chunks[i], float(scores[i]))
        for i in top_indices
        if scores[i] > 0   # skip zero-score results
    ]

In [28]:
sparse_result = sparse_search("noise bylaw requirements for construction sites", top_k=5)

In [29]:
sparse_result

[RetrievedChunk(chunk_id='noise::591-1.1::0', domain='noise', section_id='591-1.1', section_title='Definitions.', parent_section='ARTICLE 1', text="As used in this chapter, the following terms shall have the meanings indicated: AMBIENT SOUND LEVEL - The sound level that is present in the environment, produced by sound sources other than the source under assessment. AMPLIFIED SOUND - Sound made by any electronic device or a group of connected electronic devices incorporating one or more loudspeakers or other electro mechanical transducers, and intended for the production, reproduction or amplification of sound. CONSTRUCTION - Includes erection, alteration, repair, dismantling, demolition, structural maintenance, land clearing, earth-moving, grading, excavating, the laying of pipe and conduit whether above or below ground level, street and highway building, application of concrete, equipment installation and alteration and the structural installation of construction components and materi

### Hybrid Search (RRF)

In [ ]:
def hybrid_search(
    query: str,
    top_k: int = config.TOP_K_DENSE,
    domain: str | None = None,
    rrf_k: int = 60,
) -> list[RetrievedChunk]:
    dense_results = dense_search(query, top_k=top_k, domain=domain)
    sparse_results = sparse_search(query, top_k=top_k, domain=domain)

    # Build RRF score map keyed by chunk_id
    rrf_scores: dict[str, float] = {}
    chunk_map: dict[str, RetrievedChunk] = {}

    for rank, chunk in enumerate(dense_results):
        rrf_scores[chunk.chunk_id] = rrf_scores.get(chunk.chunk_id, 0) + \
            1 / (rrf_k + rank + 1)
        chunk_map[chunk.chunk_id] = chunk

    for rank, chunk in enumerate(sparse_results):
        rrf_scores[chunk.chunk_id] = rrf_scores.get(chunk.chunk_id, 0) + \
            1 / (rrf_k + rank + 1)
        chunk_map[chunk.chunk_id] = chunk

    # Sort by RRF score, return top_k
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    results = []
    for chunk_id, score in ranked:
        chunk = chunk_map[chunk_id]
        chunk.score = score
        results.append(chunk)

    return results

In [30]:
hybrid_result = hybrid_search("can I rent my apartment for airbnb?", top_k=5)

In [31]:
hybrid_result

[RetrievedChunk(chunk_id='short_term_rental::547-2.13::0', domain='short_term_rental', section_id='547-2.13', section_title='Authority to suspend a licence or registration for immediate danger.', parent_section='ARTICLE 2', text='Short-term Rental Companies', page=1, score=0.01639344262295082),
 RetrievedChunk(chunk_id='short_term_rental::547-4.1::1', domain='short_term_rental', section_id='547-4.1', section_title='Registration application and renewal requirements.', parent_section='ARTICLE 4', text="14 Editor's Note: This section of By-law 503-2024 came into force on September 30, 2024 § 547-4.1.1. Entire-unit and partial-unit rentals. [Added 2024-05-23 by By-law 503-202415] A. On an application for a registration or its renewal, the applicant shall indicate if they intend to operate an entire-unit or partial-unit rental. B. Where Municipal Licensing and Standards issues or renews a registration, the registration shall be restricted to either entire-unit or partial-unit rentals for it